# TMalign Structure Alignment

![TMalign Structure Alignment](https://proto-bio.github.io/proto-assets/images/tool/tmalign/hero.png)

This notebook demonstrates pairwise protein structure alignment using TMalign. We align two structures provided as PDB text and inspect the TM-scores, which provide a length-independent measure of topological similarity between the two inputs.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("tmalign")
display_overview("tmalign")
display_docs_section("tmalign", "Background")

# TMalign

TMalign performs pairwise protein structure alignment using the [TM-score](https://en.wikipedia.org/wiki/Template_modeling_score) metric (`tmalign-alignment`). It aligns two monomeric protein structures and returns TM-scores normalized by the length of each chain, providing a length-independent measure of structural similarity. TMalign is a compiled C++ binary that runs on CPU with no external dependencies.

**What does this tool measure/predict?**
TMalign computes the TM-score (Template Modeling score), which measures the topological similarity of two protein structures. It performs a [structural superposition](https://en.wikipedia.org/wiki/Structural_alignment) that maximizes the TM-score, then reports scores normalized by each chain's length.

**Why is this important?**
- Fold classification: TM-score > 0.5 reliably indicates proteins share the same fold topology, regardless of sequence similarity
- Structure prediction validation: compare predicted structures to experimental references
- Protein design: verify that designed proteins adopt the intended fold
- Evolutionary analysis: detect structural homologs even when sequence similarity is low (<20% identity)

**Scientific foundation:**
TM-score uses a length-dependent distance weighting scheme that emphasizes well-aligned residues and penalizes outliers less harshly than [RMSD](https://en.wikipedia.org/wiki/Root-mean-square_deviation_of_atomic_positions). The score is normalized by protein length, making it comparable across proteins of different sizes. The alignment algorithm uses an iterative heuristic that directly optimizes TM-score rather than RMSD, finding the structural superposition that maximizes topological similarity. Key properties:
- **Range:** (0, 1], where 1.0 = identical structures
- **Length-independent:** Unlike RMSD, TM-score does not increase with protein size
- **Fold discrimination:** TM-score > 0.5 reliably indicates the same fold (Zhang & Skolnick, 2004)
- **Random baseline:** Expected TM-score for randomly related proteins is ~0.17

## Available tools

In [2]:
display_available_tools("tmalign")

- **`run_tmalign()`** — Pairwise protein structure alignment using TMalign (Zhang & Skolnick, 2005). Returns TM-scores normalized by each chain.

### `run_tmalign`

TMalign performs pairwise structure alignment by iteratively superimposing two protein structures to maximize the TM-score. It accepts two structures as raw PDB text, writes them to temporary files, calls the TMalign binary, and returns two TM-scores — one normalized by the length of each chain. The score normalized by the reference chain is typically used to evaluate whether a candidate structure matches a target fold.

In [3]:
from pathlib import Path

from proto_tools import TMalignConfig, TMalignInput, TMalignOutput, run_tmalign
from proto_tools.tools.structure_alignment.tmalign.tmalign import example_input

In [4]:
# Display input docs
display_api_reference("tmalign", "input", "run_tmalign")

# Use the tool's canonical example input — a PDB aligned against itself,
# so TM-scores should come back as 1.0.
inputs = example_input()

**Input** — `TMalignInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `query_structure` | `proto_tools.entities.structures.structure.Structure` | required | Query / candidate structure (Structure object, file path, or raw PDB/CIF string) |
| `reference_structure` | `proto_tools.entities.structures.structure.Structure` | required | Reference / target structure (Structure object, file path, or raw PDB/CIF string) |

In [5]:
# Display config docs
display_api_reference("tmalign", "config", "run_tmalign")

# TMalignConfig has no tool-specific parameters; all defaults are used
config = TMalignConfig()

**Config** — `TMalignConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the alignment
result = run_tmalign(inputs, config)

Running run_tmalign [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("tmalign", "output", "run_tmalign")

print(f"TM-score (norm by Chain 1): {result.tm_score_chain_1:.3f}")
print(f"TM-score (norm by Chain 2): {result.tm_score_chain_2:.3f}")

**Output** — `TMalignOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `metrics` | `proto_tools.tools.structure_alignment.tmalign.tmalign.TMalignMetrics` | `TMalignMetrics(primary_metric=None)` | Pairwise alignment scores |

TM-score (norm by Chain 1): 1.000
TM-score (norm by Chain 2): 1.000


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("tmalign_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'tmalign_result.json'}")

Exported to example_output/tmalign_result.json
